# <font color="#32a852"> **Tamil TTS: Coqui VITS (V2 - Fixed Deps)**

**Changes in V2:**
1.  **Explicit Install:** Added `trainer` package installation which was missing.
2.  **Restart Check:** Instructions to restart runtime if imports fail.

---

### **Instructions:**
1.  Run the first cell.
2.  **IMPORTANT:** If prompted, click **"Restart Session"** (or Runtime -> Restart Session) after installation.
3.  Run the rest of the cells.

In [ ]:
#@markdown # <font color="#32a852"> **1. Setup Environment** 📦
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# Install System Deps
!sudo apt-get update -y
!sudo apt-get install -y espeak-ng libsndfile1-dev

# Install Coqui TTS & Trainer explicitly
print("Installing Coqui TTS components...")
!pip install TTS
!pip install trainer

# Fix potential numpy conflicts
!pip install "numpy<2.0"

print("✅ Environment Installed. IMPORTANT: Please restart the runtime now if this is the first run!")

In [ ]:
#@markdown # <font color="#32a852"> **2. Prepare Dataset** 📂
import os
import shutil
import zipfile

# Config
DRIVE_ROOT = "/content/drive/MyDrive"
TEXTY_ZIP = os.path.join(DRIVE_ROOT, "TextyMcSpeechy.zip")
TEXTY_DIR = os.path.join(DRIVE_ROOT, "TextyMcSpeechy")
SOURCE_DATASET = os.path.join(TEXTY_DIR, "tts_dojo/DATASETS/tamil_dataset")

# Work locally for speed
DATASET_ROOT = "/content/dataset"
if os.path.exists(DATASET_ROOT):
    shutil.rmtree(DATASET_ROOT)
os.makedirs(DATASET_ROOT)

# 1. Unzip if needed
if not os.path.exists(TEXTY_DIR):
    if os.path.exists(TEXTY_ZIP):
        print(f"Unzipping {TEXTY_ZIP}...")
        with zipfile.ZipFile(TEXTY_ZIP, 'r') as zip_ref:
            zip_ref.extractall(DRIVE_ROOT)

# 2. Copy Wavs
wav_src = os.path.join(SOURCE_DATASET, "wav_22050")
if not os.path.exists(wav_src):
    wav_src = os.path.join(SOURCE_DATASET, "wavs")

print("Copying wav files...")
target_wavs = os.path.join(DATASET_ROOT, "wavs")
shutil.copytree(wav_src, target_wavs)

# 3. Format Metadata (LJSpeech format)
# Source: filename|text
# Target: filename|text|text (Coqui likes this format)
print("Formatting metadata...")
meta_src = os.path.join(SOURCE_DATASET, "metadata.csv")
meta_target = os.path.join(DATASET_ROOT, "metadata.csv")

with open(meta_src, 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open(meta_target, 'w', encoding='utf-8') as f:
    for line in lines:
        parts = line.strip().split('|')
        if len(parts) >= 2:
            filename = parts[0]
            text = parts[-1]
            f.write(f"{filename}|{text}|{text}\n")

print(f"✅ Dataset ready at {DATASET_ROOT}")

In [ ]:
#@markdown # <font color="#32a852"> **3. Configure Training (VITS)** ⚙️

try:
    from trainer import Trainer, TrainerArgs
except ImportError:
    raise ImportError("❌ Module 'trainer' not found. Please Restart Runtime (Runtime -> Restart Session) and run cells again!")

from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.configs.shared_configs import BaseDatasetConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.vits import Vits
from TTS.utils.audio import AudioProcessor
from TTS.tts.utils.text.tokenizer import TTSTokenizer

# Output Path
OUT_PATH = "/content/drive/MyDrive/coqui_vits_tamil"

# CONFIGURATION
dataset_config = BaseDatasetConfig(
    formatter="ljspeech",
    meta_file_train="metadata.csv",
    path=DATASET_ROOT
)

config = VitsConfig(
    batch_size=8,
    eval_batch_size=4,
    batch_group_size=4,
    num_loader_workers=2,
    num_eval_loader_workers=2,
    run_eval=True,
    test_delay_epochs=-1,
    epochs=100,
    text_cleaner="multilingual_cleaner",
    use_phonemes=True,
    phoneme_language="ta", # TAMIL
    phoneme_cache_path=os.path.join(OUT_PATH, "phoneme_cache"),
    compute_input_seq_cache=True,
    print_step=25,
    print_eval=True,
    mixed_precision=True,
    output_path=OUT_PATH,
    datasets=[dataset_config]
)

# Initialize Audio Processor
ap = AudioProcessor.init_from_config(config)

# Tokenizer (Required for VITS)
tokenizer = TTSTokenizer.init_from_config(config)

# Load Samples
train_samples, eval_samples = load_tts_samples(
    dataset_config,
    eval_split=True,
    eval_split_max_size=dataset_config.eval_split_max_size,
    eval_split_size=dataset_config.eval_split_size,
)

# Initialize Model
model = Vits(config, ap, tokenizer, speaker_manager=None)

print("✅ Configuration Setup Complete!")

In [ ]:
#@markdown # <font color="#32a852"> **4. Start Training** 🚀

# Initialize Trainer
trainer = Trainer(
    TrainerArgs(),
    config,
    OUT_PATH,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

# START
trainer.fit()